# Préparation de l'espace de travail

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
data_complet = pd.read_parquet("data/data_complet.parquet")

# Graphiques

In [ ]:
def graph_evol_lic_age(df, age):
    """
    Affiche un graphique interactif de l'évolution des effectifs de licenciés par âge et par sport.

    Paramètres
    ----------
    df : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'sport', 'age', 'licences_annuelles', 'code_sport'].
    age : str ou "all", optionnel
        Filtre pour un âge spécifique. Si "all", affiche tous les âges.

    Output
    ------
    Graphique interactif Plotly représentant le nombre de licenciés par sport et par année pour l'âge sélectionné.
    """
    
    # Suppression des codes sport non pertinents
    df_clean = df[df["code_sport"] != "DIV"]
    
    # Filtrage par âge si demandé
    if age != "all":
        df_filtre = df_clean[df_clean["age"] == age]
        titre_age = f"{age} ans"
    else:
        df_filtre = df_clean.copy()
        titre_age = "tous les âges"

    # Agrégation des licences par année et par sport
    table = (
        df_filtre.groupby(["annee", "sport"])["licences_annuelles"]
        .sum()
        .unstack()
        .sort_index()
    )

    # Transformation du DataFrame pour le format "long" (pour Plotly)
    table_long = table.reset_index().melt(
        id_vars="annee",
        var_name="sport",
        value_name="licences_annuelles"
    )

    # Tracé
    fig = px.line(
        table_long,
        x="annee",
        y="licences_annuelles",
        color="sport",
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        markers=True,
        labels={
            "annee": "Année",
            "licences_annuelles": "Nombre de licenciés",
            "sport": "Sport"
        },
        title=f"Évolution du nombre de licenciés de {titre_age} par sport"
    )

    fig.update_layout(width=1100, height=600)
    fig.show()

In [ ]:
# Code interactif

# Préparation des options du widget
# Tri d'abord les âges numériques
ages_numeric = sorted([a for a in data_complet["age"].dropna().unique() if a not in ["NR - Non réparti"]], key=int)

# Ajout de "NR - Non réparti" à la fin si présent
ages = ages_numeric
if "NR - Non réparti" in data_complet["age"].unique():
    ages.append("NR - Non réparti")

# # Création du widget Dropdown
age_widget = widgets.Dropdown(options=["all"] + ages, description="Age :", value="all")

# Widget de sortie (utile pour afficher le graphique dans le notebook)
out = widgets.Output()

# Fonction de mise à jour du graphique
def update_graph(change=None):
    """
    Met à jour le graphique interactif lorsque l'utilisateur change l'âge sélectionné.
    """
    clear_output(wait=True)
    display(age_widget)
    
    selected_age = age_widget.value
    graph_evol_lic_age(data_complet, age=selected_age)

# Liaison du widget à la fonction de mise à jour
age_widget.observe(update_graph, names='value')

# Affichage initial
display(age_widget, out)
update_graph()


In [ ]:
def graph_evol_lic_tranche_fine_age(df, tranche="all"):
    """
    Affiche un graphique interactif de l'évolution des effectifs de licenciés 
    par tranche d'âge fine et par sport.

    Paramètres
    ----------
    df : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'sport', 'tranche_age', 'licences_annuelles', 'code_sport'].
    tranche : str ou "all", optionnel
        Filtre pour une tranche d'âge spécifique. Si "all", affiche toutes les tranches.

    Output
    ------
    Graphique interactif Plotly représentant le nombre de licenciés par sport et par année pour la tranche d'âge sélectionnée.
    """
    
    # Suppression des codes sportifs non pertinents
    df_clean = df[df["code_sport"] != "DIV"]

    # Filtrage par tranche d'âge si demandé
    if tranche != "all":
        df_filtre = df_clean[df_clean["tranche_age"] == tranche]
        # Extraction du libellé de la tranche pour le titre
        tranche_label = df_filtre["tranche_age"].str[4:].unique()[0]
        titre_age = f"{tranche_label}"
    else:
        df_filtre = df_clean.copy()
        titre_age = "de toutes les tranches d'âge"

    # Tri par année et par sport pour l'agrégation
    df_filtre = df_filtre.sort_values(["annee", "sport"])

    # Agrégation des licences par année et par sport
    table = (
        df_filtre.groupby(["annee", "sport"])["licences_annuelles"]
        .sum()
        .unstack()
        .sort_index()
    )

    # Transformation du DataFrame pour le format "long"
    table_long = table.reset_index().melt(
        id_vars="annee",
        var_name="sport",
        value_name="licences_annuelles"
    )

    # Tracé interactif
    fig = px.line(
        table_long,
        x="annee",
        y="licences_annuelles",
        color="sport",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        labels={
            "annee": "Année",
            "licences_annuelles": "Nombre de licenciés",
            "sport": "Sport"
        },
        title=f"Évolution du nombre de licenciés {titre_age} par sport"
    )

    # Mise en forme du graphique
    fig.update_layout(width=1200, height=650)
    fig.show()


In [ ]:
# Code interactif

# Préparation des options du widget
# Récupération et tri des tranches d'âge uniques
ages = sorted(data_complet["tranche_age"].dropna().unique())

# Ajout de l'option "all" pour visualiser toutes les tranches
options = ["all"] + list(ages)

# Création du widget Dropdown
age_widget = widgets.Dropdown(options=options, description="Tranche :", value="all")

# Widget de sortie pour afficher le graphique
out = widgets.Output()

# Fonction de mise à jour du graphique
def update_graph(change=None):
    """
    Met à jour le graphique interactif lorsque l'utilisateur change la tranche sélectionnée.
    """
    clear_output(wait=True)
    display(age_widget)
    
    selected_tranche = age_widget.value
    graph_evol_lic_tranche_fine_age(data_complet, tranche=selected_tranche)

# Liaison du widget à la fonction de mise à jour
age_widget.observe(update_graph, names='value')

# Affichage initial
display(age_widget, out)
update_graph()

In [ ]:
def graph_decompo_sports_tranche_grande(df, annee="all"):
    """
    Affiche un graphique décomposant les effectifs de licenciés par grande tranche d'âge 
    et par sport pour une année donnée.

    Paramètres
    ----------
    df : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'sport', 'grande_tranche_age', 'licences_annuelles'].
    annee : int ou "all", optionnel
        Année sélectionnée. Si "all", affiche toutes les années.

    Output
    ------
    Graphique interactif Plotly représentant la répartition proportionnelle des licenciés 
    par grande tranche d'âge pour chaque sport.
    """
    
    # Filtrage selon l'année
    if annee == "all":
        df_clean = df.copy()
        titre_annee = "2016 - 2024"
    else:
        df_clean = df[df["annee"] == annee]
        titre_annee = annee

    # Pivot pour obtenir les effectifs par sport et grande tranche d'âge
    df_pivot = df_clean.pivot_table(
        index='sport',
        columns='grande_tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    # Conversion en proportions par sport
    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0)

    # Transformation en format long pour Plotly
    df_long = df_prop.reset_index().melt(
        id_vars="sport",
        var_name="grande_tranche_age",
        value_name="proportion"
    )

    # Conversion en pourcentage
    df_long["proportion"] = df_long["proportion"] * 100

    # Construction du mapping des couleurs
    tranches = sorted(df_long["grande_tranche_age"].unique())

    palette_map = {}
    if "NR - Non réparti" in tranches:
        tranches_no_nr = [t for t in tranches if t != "NR - Non réparti"]
    else:
        tranches_no_nr = tranches

    n = len(tranches_no_nr)
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Plasma_r,
        [i/(n-1) for i in range(n)] if n > 1 else [0.5]
    )

    # Assigner les couleurs aux tranches (hors NR)
    for tranche, col in zip(tranches_no_nr, colors):
        palette_map[tranche] = col

    # Assigner NR en noir si présent
    if "NR - Non réparti" in tranches:
        palette_map["NR - Non réparti"] = "black"

    # Création du graphique
    fig = px.bar(
        df_long,
        x="proportion",
        y="sport",
        color="grande_tranche_age",
        color_discrete_map=palette_map,
        orientation="h",
        barmode="stack",
        labels={
            "proportion": "Proportion de licenciés (%)",
            "sport": "Sport",
            "grande_tranche_age": "Tranche d'âge"
        },
        title=f"Répartition proportionnelle des licenciés par sport et grande tranche d'âge – {titre_annee}"
    )

    # Mise en forme de l'axe x et dimensions
    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(
        width=1000,
        height=800,
        xaxis=dict(range=[0, 100])
    )

    # Affichage
    fig.show()

In [ ]:
# Code interactif

# Préparation des options du widget
# Récupération des années uniques et triées
annees = sorted(data_complet["annee"].dropna().unique())

# Ajout de l'option "all" pour visualiser toutes les années
options = ["all"] + list(annees)

# Création du widget Dropdown
annees_widget = widgets.Dropdown(
    options=options,
    description="Années :",
    value="all"
)

# Widget de sortie pour afficher le graphique
out = widgets.Output()

# Fonction de mise à jour du graphique
def update_graph(change=None):
    """
    Met à jour le graphique interactif lorsque l'utilisateur change l'année sélectionnée.
    """
    clear_output(wait=True)
    display(annees_widget)
    
    selected_annee = annees_widget.value
    graph_decompo_sports_tranche_grande(data_complet, annee=selected_annee)

# Liaison du widget à la fonction de mise à jour
annees_widget.observe(update_graph, names='value')

# Affichage initial
display(annees_widget, out)
update_graph()

In [ ]:
def graph_decompo_sports_tranche_fine(df, annee="all"):
    """
    Affiche un graphique décomposant les effectifs de licenciés par tranche d'âge fine 
    et par sport pour une année donnée.

    Paramètres
    ----------
    df : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'sport', 'tranche_age', 'licences_annuelles'].
    annee : int ou "all", optionnel
        Année sélectionnée. Si "all", affiche toutes les années.

    Output
    ------
    Graphique interactif Plotly représentant la répartition proportionnelle des licenciés 
    par tranche d'âge fine pour chaque sport.
    """
    
    # Filtrage selon l'année
    if annee == "all":
        df_clean = df.copy()
        titre_annee = "2016 - 2024"
    else:
        df_clean = df[df["annee"] == annee]
        titre_annee = annee

    # Pivot pour obtenir les effectifs par sport et tranche d'âge fine
    df_pivot = df_clean.pivot_table(
        index='sport',
        columns='tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    # Conversion en proportions par sport
    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0).fillna(0)

    # Transformation en format long pour Plotly
    df_long = df_prop.reset_index().melt(
        id_vars="sport",
        var_name="tranche_age",
        value_name="proportion"
    )

    # Suppression des tranches vides ou NaN et conversion en str
    df_long = df_long[df_long["tranche_age"].notna()]
    df_long["tranche_age"] = df_long["tranche_age"].astype(str)

    # Conversion en pourcentage
    df_long["proportion"] = df_long["proportion"] * 100

    # Construction des couleurs
    tranches = sorted(df_long["tranche_age"].unique())
    tranches_no_nr = [t for t in tranches if t != "NR - Non réparti"]
    n = len(tranches_no_nr)

    # Générer n couleurs dans Plasma_r
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Plasma_r,
        [i/(n-1) for i in range(n)] if n > 1 else [0.5]
    )

    # Mapping des couleurs
    palette_map = {t: c for t, c in zip(tranches_no_nr, colors)}
    if "NR - Non réparti" in tranches:
        palette_map["NR - Non réparti"] = "black"

    # Définition de l'ordre des tranches pour le graphique
    tranches_ord = sorted(tranches_no_nr)
    if "NR - Non réparti" in tranches:
        tranches_ord.append("NR - Non réparti")

    # Création du graphique
    fig = px.bar(
        df_long,
        x="proportion",
        y="sport",
        color="tranche_age",
        color_discrete_map=palette_map,
        category_orders={"tranche_age": tranches_ord},
        orientation="h",
        barmode="stack",
        labels={
            "proportion": "Proportion de licenciés (%)",
            "sport": "Sport",
            "tranche_age": "Tranche d'âge"
        },
        title=f"Répartition proportionnelle des licenciés par sport et tranche d'âge fine – {titre_annee}"
    )

    # Axe X en pourcentage et mise en forme
    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(
        width=1000,
        height=800,
        xaxis=dict(range=[1, 100])
    )

    # Affichage
    fig.show()

In [ ]:
# Code interactif

# Préparation des options du widget
# Récupération des années uniques et triées
annees = sorted(data_complet["annee"].dropna().unique())

# Ajout de l'option "all" pour visualiser toutes les années
options = ["all"] + list(annees)

# Création du widget Dropdown
annees_widget = widgets.Dropdown(
    options=options,
    description="Années :",
    value="all"
)

# Widget de sortie pour afficher le graphique
out = widgets.Output()

# Fonction de mise à jour du graphique
def update_graph(change=None):
    """
    Met à jour le graphique interactif lorsque l'utilisateur change l'année sélectionnée.
    """
    clear_output(wait=True)
    display(annees_widget)
    
    selected_annee = annees_widget.value
    graph_decompo_sports_tranche_fine(data_complet, annee=selected_annee)

# Liaison du widget à la fonction de mise à jour
annees_widget.observe(update_graph, names='value')

# Affichage initial
display(annees_widget, out)
update_graph()